# SDXL Compression Study

Systematic optimization of Stable Diffusion XL.

> Rajat Gupta · Pruna AI Technical Interview · April 2026

**All results are saved to Google Drive — survives disconnects.**

In [ ]:
# ============================================================
# CELL 0: Mount Drive + Clone + Install (run every reconnect)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Persistent results directory on Drive
RESULTS_DIR = '/content/drive/MyDrive/sdxl-optimization-results'
!mkdir -p {RESULTS_DIR}

import os
os.chdir('/content')

REPO_URL = 'https://github.com/rajat116/sdxl-optimization.git'
if not os.path.exists('/content/sdxl-optimization/src'):
    !git clone {REPO_URL}
else:
    os.chdir('/content/sdxl-optimization')
    !git fetch origin && git reset --hard origin/main
    print('✅ Pulled latest from GitHub')

os.chdir('/content/sdxl-optimization')

# Symlink results dir to Drive so all saves persist
!rm -rf results
!ln -s {RESULTS_DIR} results

!pip install -q -e '.[dev]'
!pip install -q DeepCache
print('\n✅ Setup complete. Results persist in Google Drive.')

In [ ]:
# ============================================================
# CELL 1: Imports + GPU check
# ============================================================
import sys, logging, gc, time, json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage

sys.path.insert(0, 'src')

from sdxl_opt.utils import (
    setup_logging, seed_everything, get_gpu_info,
    reset_peak_memory, gpu_peak_memory_gb, EVAL_PROMPTS, timer
)
from sdxl_opt.pipeline import CompressionConfig, load_pipeline, generate_images
from sdxl_opt.evaluate import compute_clip_scores, save_comparison_grid, save_individual_images
from sdxl_opt.pareto import plot_pareto, plot_speedup_bar, plot_memory_comparison, find_pareto_frontier

setup_logging('INFO')
gpu = get_gpu_info()
print(f'\n✅ GPU: {gpu}')

In [ ]:
# ============================================================
# CELL 2: Checkpoint system (loads from Drive)
# ============================================================
CHECKPOINT = 'results/checkpoint.json'

def save_checkpoint(results, section):
    with open(CHECKPOINT, 'w') as f:
        json.dump({'results': results, 'section': section}, f)
    print(f'💾 Saved checkpoint after {section} ({len(results)} configs done) → Google Drive')

def load_checkpoint():
    try:
        with open(CHECKPOINT) as f:
            data = json.load(f)
        print(f'📂 Loaded checkpoint: {data["section"]} ({len(data["results"])} configs done)')
        return data['results'], data['section']
    except FileNotFoundError:
        print('No checkpoint found, starting fresh')
        return [], ''

results, last_section = load_checkpoint()
all_images = {}
prompts = EVAL_PROMPTS[:4]

def already_done(name):
    return any(r['config'] == name for r in results)

print(f'Configs already completed: {[r["config"] for r in results]}')

In [ ]:
# ============================================================
# CELL 3: Benchmark function
# ============================================================
def benchmark_one(config, prompts, n_runs=3, n_warmup=2):
    pipe = load_pipeline(config)
    gen = seed_everything(0)
    for _ in range(n_warmup):
        generate_images(pipe, config, ['warmup'], generator=gen, height=512, width=512)
    torch.cuda.synchronize()

    latencies = []
    last_imgs = []
    for run in range(n_runs):
        gen = seed_everything(42)
        reset_peak_memory()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        imgs = generate_images(pipe, config, prompts, generator=gen)
        torch.cuda.synchronize()
        per_img = (time.perf_counter() - t0) / len(prompts)
        latencies.append(per_img)
        last_imgs = imgs
        print(f'  Run {run+1}/{n_runs}: {per_img:.2f}s/img | Peak VRAM: {gpu_peak_memory_gb():.2f} GB')

    result = {
        'config': config.name, 'label': config.short_label(),
        'steps': config.num_inference_steps,
        'latency_mean_s': round(float(np.mean(latencies)), 3),
        'latency_std_s': round(float(np.std(latencies)), 3),
        'peak_vram_gb': round(gpu_peak_memory_gb(), 2),
    }

    # Save images to Drive
    for i, img in enumerate(last_imgs):
        img_dir = Path(f'results/images/{config.name}')
        img_dir.mkdir(parents=True, exist_ok=True)
        img.save(img_dir / f'sample_{i:02d}.png')

    del pipe; gc.collect(); torch.cuda.empty_cache()
    return result, last_imgs

print('✅ Benchmark function ready')

## 1. Profiling

In [ ]:
if not already_done('baseline'):
    baseline_cfg = CompressionConfig(name='baseline', dtype='fp16', num_inference_steps=50, guidance_scale=7.5)
    pipe = load_pipeline(baseline_cfg)
    gen = seed_everything(42)
    with timer() as t_enc:
        _ = pipe.encode_prompt(prompt='a test image', device='cuda')
    print(f'Text encoding: {t_enc["elapsed_s"]:.3f}s')

    reset_peak_memory()
    gen = seed_everything(42)
    with timer() as t_full:
        img = pipe('a test image', num_inference_steps=50, guidance_scale=7.5, generator=gen).images[0]
    print(f'\nBaseline: {t_full["elapsed_s"]:.2f}s | Peak VRAM: {gpu_peak_memory_gb():.2f} GB')
    del pipe; gc.collect(); torch.cuda.empty_cache()
else:
    print('Baseline already profiled')

## 2. Axis-by-Axis Compression

In [ ]:
# 2.1 Baseline
if not already_done('baseline'):
    cfg = CompressionConfig(name='baseline', dtype='fp16', num_inference_steps=50, guidance_scale=7.5)
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, '2.1_baseline')
else:
    print('✓ baseline done')

In [ ]:
# 2.2 Quantization
for cfg in [
    CompressionConfig(name='int8_unet', dtype='fp16', quantize_unet='int8', num_inference_steps=50, guidance_scale=7.5),
    CompressionConfig(name='nf4_unet', dtype='fp16', quantize_unet='nf4', num_inference_steps=50, guidance_scale=7.5),
]:
    if already_done(cfg.name):
        print(f'✓ {cfg.name} done'); continue
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, f'2.2_{cfg.name}')

In [ ]:
# 2.3 DeepCache
for cfg in [
    CompressionConfig(name='deepcache_2', dtype='fp16', use_deepcache=True, deepcache_interval=2, num_inference_steps=50, guidance_scale=7.5),
    CompressionConfig(name='deepcache_3', dtype='fp16', use_deepcache=True, deepcache_interval=3, num_inference_steps=50, guidance_scale=7.5),
]:
    if already_done(cfg.name):
        print(f'✓ {cfg.name} done'); continue
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, f'2.3_{cfg.name}')

In [ ]:
# 2.4 LCM-LoRA
for cfg in [
    CompressionConfig(name='lcm_8step', dtype='fp16', use_lcm_lora=True, num_inference_steps=8, guidance_scale=1.5),
    CompressionConfig(name='lcm_4step', dtype='fp16', use_lcm_lora=True, num_inference_steps=4, guidance_scale=1.5),
]:
    if already_done(cfg.name):
        print(f'✓ {cfg.name} done'); continue
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, f'2.4_{cfg.name}')

In [ ]:
# 2.5 torch.compile
if not already_done('compiled'):
    cfg = CompressionConfig(name='compiled', dtype='fp16', use_torch_compile=True, compile_mode='reduce-overhead', num_inference_steps=50, guidance_scale=7.5)
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3, n_warmup=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, '2.5_compiled')
else:
    print('✓ compiled done')

In [ ]:
# 2.6 Tiny VAE
if not already_done('tiny_vae'):
    cfg = CompressionConfig(name='tiny_vae', dtype='fp16', use_tiny_vae=True, num_inference_steps=50, guidance_scale=7.5)
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, '2.6_tiny_vae')
else:
    print('✓ tiny_vae done')

## 3. Stacked Combinations

In [ ]:
combo_configs = [
    CompressionConfig(name='nf4+deepcache', dtype='fp16', quantize_unet='nf4',
                      use_deepcache=True, deepcache_interval=2,
                      num_inference_steps=50, guidance_scale=7.5),
    CompressionConfig(name='lcm+deepcache', dtype='fp16',
                      use_lcm_lora=True, use_deepcache=True, deepcache_interval=2,
                      num_inference_steps=8, guidance_scale=1.5),
    CompressionConfig(name='lcm+compiled+tvae', dtype='fp16',
                      use_lcm_lora=True, use_torch_compile=True,
                      use_tiny_vae=True, num_inference_steps=8, guidance_scale=1.5),
    # NOTE: full_stack with compile+deepcache crashes (CUDA graph conflict).
    # We use a no-compile version instead.
    CompressionConfig(name='full_stack_nocompile', dtype='fp16',
                      quantize_unet='nf4', use_lcm_lora=True,
                      use_deepcache=True, deepcache_interval=2,
                      use_tiny_vae=True,
                      num_inference_steps=8, guidance_scale=1.5),
]

for cfg in combo_configs:
    if already_done(cfg.name):
        print(f'✓ {cfg.name} done'); continue
    print(f'\n=== {cfg.name} ===')
    try:
        r, imgs = benchmark_one(cfg, prompts, n_runs=3, n_warmup=3)
        results.append(r); all_images[cfg.name] = imgs
        print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
        save_checkpoint(results, f'3_{cfg.name}')
    except Exception as e:
        print(f'  ❌ FAILED: {e}')
        gc.collect(); torch.cuda.empty_cache()

## 4. Quality Evaluation (CLIP Score)

In [ ]:
df = pd.DataFrame(results)

# For configs with images in memory, compute CLIP
for cfg_name, imgs in all_images.items():
    print(f'CLIP score for {cfg_name}...')
    mean, std, scores = compute_clip_scores(imgs, prompts[:len(imgs)])
    df.loc[df['config'] == cfg_name, 'clip_score'] = mean
    df.loc[df['config'] == cfg_name, 'clip_score_std'] = std

# For configs from checkpoint (no images in memory), reload from Drive
missing = df[df['clip_score'].isna()]['config'].tolist()
if missing:
    print(f'\nReloading images from Drive for: {missing}')
    from PIL import Image
    for cfg_name in missing:
        img_dir = Path(f'results/images/{cfg_name}')
        if img_dir.exists():
            imgs = sorted(img_dir.glob('*.png'))
            if imgs:
                loaded = [Image.open(p) for p in imgs[:len(prompts)]]
                print(f'  CLIP score for {cfg_name} (from disk)...')
                mean, std, _ = compute_clip_scores(loaded, prompts[:len(loaded)])
                df.loc[df['config'] == cfg_name, 'clip_score'] = mean
                df.loc[df['config'] == cfg_name, 'clip_score_std'] = std
        else:
            print(f'  ⚠️ No images found for {cfg_name}')

# Speedup
baseline_lat = df.loc[df['config'] == 'baseline', 'latency_mean_s'].values
if len(baseline_lat) > 0:
    df['speedup'] = (baseline_lat[0] / df['latency_mean_s']).round(2)

print('\n' + '='*90)
display(df.sort_values('speedup', ascending=False) if 'speedup' in df.columns else df)

df.to_csv('results/benchmark_results.csv', index=False)
print('\n💾 Saved to Google Drive')

## 5. Visualizations

In [ ]:
if 'clip_score' in df.columns and df['clip_score'].notna().any():
    plot_pareto(df[df['clip_score'].notna()], 'results/pareto_frontier.png')
    display(IPImage('results/pareto_frontier.png', width=800))

plot_speedup_bar(df, output_path='results/speedup_bar.png')
display(IPImage('results/speedup_bar.png', width=700))

plot_memory_comparison(df, output_path='results/memory_comparison.png')
display(IPImage('results/memory_comparison.png', width=700))

if all_images:
    save_comparison_grid(all_images, prompts, 'results/comparison_grid.png', max_prompts=4)
    display(IPImage('results/comparison_grid.png', width=1000))

print('💾 All plots saved to Google Drive')

## 6. Pareto-Optimal Recommendations

In [ ]:
if 'clip_score' in df.columns and df['clip_score'].notna().any():
    pareto_df = find_pareto_frontier(df[df['clip_score'].notna()], 'latency_mean_s', 'clip_score')
    print('Pareto-optimal configurations:')
    display(pareto_df[['config', 'latency_mean_s', 'peak_vram_gb', 'clip_score', 'speedup']].sort_values('speedup', ascending=False))

print('\n📋 Deployment Recommendations:')
print('  🚀 Speed:    lcm_4step — 6.7× faster, no quality loss')
print('  ⚖️  Balanced: lcm+compiled+tvae — 5.5× faster, 25% less VRAM')
print('  🎨 Quality:  deepcache_2 — 1.7× faster, zero quality degradation')

## 7. Deployment Demo

In [ ]:
# Direct inference demo (more reliable than LitServe on free Colab)
cfg = CompressionConfig(name='speed_demo', dtype='fp16', use_lcm_lora=True,
                        num_inference_steps=4, guidance_scale=1.5)
pipe = load_pipeline(cfg)
gen = seed_everything(42)

# Warmup
_ = generate_images(pipe, cfg, ['warmup'], generator=seed_everything(0), height=512, width=512)

# Timed inference
gen = seed_everything(42)
t0 = time.perf_counter()
imgs = generate_images(pipe, cfg, ['a photo of an astronaut riding a horse on mars, cinematic lighting'], generator=gen)
latency = time.perf_counter() - t0

print(f'🚀 Optimized inference: {latency:.2f}s (vs ~5.7s baseline = {5.66/latency:.1f}× speedup)')
imgs[0].save('results/deployment_demo.png')
display(imgs[0])

del pipe; gc.collect(); torch.cuda.empty_cache()

## 8. Summary & Future Work

### Key Findings
- **Highest leverage:** LCM-LoRA step reduction (50 → 4 steps) gives 6.7× speedup with no quality loss
- **Best balanced:** LCM + torch.compile + TinyVAE — 5.5× faster, 25% less VRAM
- **Memory champion:** NF4 quantization saves ~3.4 GB but is slower on T4 (no INT4 tensor cores)
- **Negative results:** torch.compile has huge warmup variance on T4; DeepCache + CUDA graphs conflict; not all axes compose

### Ideas for Future Work
- Test on A100 where quantization speedup is expected to be positive
- Ternary quantization (BitNet-style) for the UNet
- Per-block mixed precision based on sensitivity analysis
- Speculative decoding for diffusion — small model for early steps
- Pruna integration — wrap all methods into `smash()` calls

In [ ]:
# Download results
!cd results && zip -r /content/results.zip .
from google.colab import files
files.download('/content/results.zip')

---
## 8. HQQ Quantization — The Right Quantizer

**Why we revisit quantization here:**

In Section 2.2, BitsAndBytes INT8/NF4 made inference *slower* on A100. The root cause:
BitsAndBytes stores weights compressed but **dequantizes back to FP16 before every matmul** — that overhead dominates on fast GPUs.

**HQQ (Half-Quadratic Quantization) with Marlin backend** avoids this entirely:
it runs **native INT4 matrix multiplies directly on GPU** — no dequantization step.

| Quantizer | Inference mechanism | Expected on A100 |
|-----------|--------------------|-----------------|
| BitsAndBytes NF4 | Store INT4 → dequant to FP16 → matmul | ❌ Slower (overhead dominates) |
| **HQQ 4-bit + Marlin** | **Native INT4 GEMM kernel** | ✅ Faster + ~60% less VRAM |

> This is exactly the quantizer Pruna AI uses in their `smash()` API for diffusion models.

**Strategy:** We load results from Drive checkpoint, skip already-done configs, only run new HQQ experiments.

In [ ]:
# ============================================================
# CELL 8.0: Install HQQ + reload checkpoint from Drive
# (Skip if already installed / checkpoint already loaded)
# ============================================================
!pip install -q hqq pruna

# Reload checkpoint in case this section is run fresh
import json, gc, time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Image as IPImage

CHECKPOINT = 'results/checkpoint.json'

def load_checkpoint():
    try:
        with open(CHECKPOINT) as f:
            data = json.load(f)
        print(f'📂 Loaded checkpoint: {data["section"]} ({len(data["results"])} configs done)')
        return data['results'], data['section']
    except FileNotFoundError:
        print('No checkpoint found — run Section 1–7 first or mount Drive')
        return [], ''

def save_checkpoint(results, section):
    with open(CHECKPOINT, 'w') as f:
        json.dump({'results': results, 'section': section}, f)
    print(f'💾 Checkpoint saved after {section} ({len(results)} configs total)')

def already_done(name):
    return any(r['config'] == name for r in results)

try:
    results
    print(f'Results already in memory: {len(results)} configs')
except NameError:
    results, last_section = load_checkpoint()

prompts = [
    'a photo of an astronaut riding a horse on mars, cinematic lighting',
    'a beautiful sunset over a calm ocean with sailboats, oil painting',
    'a corgi wearing a top hat and monocle, portrait photography',
    'a futuristic cityscape at night with neon lights, cyberpunk',
]

print(f'\nConfigs already done: {[r["config"] for r in results]}')
print('\n✅ Ready for HQQ experiments')

In [ ]:
# ============================================================
# CELL 8.1: HQQ standalone benchmark
# ============================================================
import sys
sys.path.insert(0, 'src')
from sdxl_opt.pipeline import CompressionConfig, load_pipeline, generate_images
from sdxl_opt.utils import seed_everything, reset_peak_memory, gpu_peak_memory_gb

def benchmark_hqq(config, prompts, n_runs=3, n_warmup=2):
    pipe = load_pipeline(config)
    gen = seed_everything(0)
    for _ in range(n_warmup):
        generate_images(pipe, config, ['warmup'], generator=gen, height=512, width=512)
    torch.cuda.synchronize()

    latencies, last_imgs = [], []
    for run in range(n_runs):
        gen = seed_everything(42)
        reset_peak_memory()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        imgs = generate_images(pipe, config, prompts, generator=gen)
        torch.cuda.synchronize()
        per_img = (time.perf_counter() - t0) / len(prompts)
        latencies.append(per_img)
        last_imgs = imgs
        print(f'  Run {run+1}/{n_runs}: {per_img:.2f}s/img | Peak VRAM: {gpu_peak_memory_gb():.2f} GB')

    result = {
        'config': config.name, 'label': config.short_label(),
        'steps': config.num_inference_steps,
        'latency_mean_s': round(float(np.mean(latencies)), 3),
        'latency_std_s': round(float(np.std(latencies)), 3),
        'peak_vram_gb': round(gpu_peak_memory_gb(), 2),
    }

    for i, img in enumerate(last_imgs):
        img_dir = Path(f'results/images/{config.name}')
        img_dir.mkdir(parents=True, exist_ok=True)
        img.save(img_dir / f'sample_{i:02d}.png')

    del pipe; gc.collect(); torch.cuda.empty_cache()
    return result, last_imgs

# --- HQQ 4-bit standalone ---
if not already_done('hqq_4bit'):
    cfg = CompressionConfig(
        name='hqq_4bit', dtype='fp16',
        use_hqq=True, hqq_weight_bits=4, hqq_group_size=64,
        num_inference_steps=50, guidance_scale=7.5
    )
    print(f'\n=== {cfg.name} ===')
    print('Testing HQQ 4-bit quantization (no dequant overhead — native INT4 kernel)')
    r, imgs = benchmark_hqq(cfg, prompts, n_runs=3)
    results.append(r)
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    baseline_lat = next((x['latency_mean_s'] for x in results if x['config'] == 'baseline'), 5.66)
    print(f'  → Speedup vs baseline: {baseline_lat / r["latency_mean_s"]:.2f}×')
    save_checkpoint(results, '8.1_hqq_4bit')
else:
    print('✓ hqq_4bit already done')

In [ ]:
# ============================================================
# CELL 8.2: HQQ 4-bit + DeepCache
# This is the Pruna-style recommended combo for SDXL U-Net models
# ============================================================
!pip install -q DeepCache

if not already_done('hqq_4bit+deepcache'):
    cfg = CompressionConfig(
        name='hqq_4bit+deepcache', dtype='fp16',
        use_hqq=True, hqq_weight_bits=4, hqq_group_size=64,
        use_deepcache=True, deepcache_interval=2,
        num_inference_steps=50, guidance_scale=7.5
    )
    print(f'\n=== {cfg.name} ===')
    print('HQQ 4-bit (memory ↓) + DeepCache (skip UNet blocks, latency ↓)')
    print('This is orthogonal: HQQ reduces per-op cost, DeepCache reduces op count')
    r, imgs = benchmark_hqq(cfg, prompts, n_runs=3)
    results.append(r)
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    baseline_lat = next((x['latency_mean_s'] for x in results if x['config'] == 'baseline'), 5.66)
    print(f'  → Speedup vs baseline: {baseline_lat / r["latency_mean_s"]:.2f}×')
    save_checkpoint(results, '8.2_hqq+deepcache')
else:
    print('✓ hqq_4bit+deepcache already done')

In [ ]:
# ============================================================
# CELL 8.3: HQQ 4-bit + torch.compile
# torch.compile further optimizes the quantized INT4 kernel graph
# ============================================================
if not already_done('hqq_4bit+compile'):
    cfg = CompressionConfig(
        name='hqq_4bit+compile', dtype='fp16',
        use_hqq=True, hqq_weight_bits=4, hqq_group_size=64,
        use_torch_compile=True, compile_mode='reduce-overhead',
        num_inference_steps=50, guidance_scale=7.5
    )
    print(f'\n=== {cfg.name} ===')
    print('HQQ 4-bit + torch.compile — extra warmup runs needed (graph capture)')
    r, imgs = benchmark_hqq(cfg, prompts, n_runs=3, n_warmup=4)
    results.append(r)
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    baseline_lat = next((x['latency_mean_s'] for x in results if x['config'] == 'baseline'), 5.66)
    print(f'  → Speedup vs baseline: {baseline_lat / r["latency_mean_s"]:.2f}×')
    save_checkpoint(results, '8.3_hqq+compile')
else:
    print('✓ hqq_4bit+compile already done')

In [ ]:
# ============================================================
# CELL 8.4: Pruna smash() — their one-call API
# Pruna's recommended combo for SDXL: hqq_diffusers + deepcache
#
# FIX: Pruna conflicts with the transformers version already loaded
# in this session. We pin a compatible version, then use a fresh
# subprocess so we don't need to restart the kernel manually.
# ============================================================

# Step 1 — pin compatible transformers BEFORE importing pruna
# (pruna needs transformers <4.47, Colab default is newer)
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'transformers>=4.40.0,<4.47.0', 'pruna', '--upgrade'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else '')
if result.returncode != 0:
    print("pip error:", result.stderr[-300:])
else:
    print("✅ transformers pinned, pruna installed")

# Step 2 — run the smash experiment in a subprocess so the pinned
# transformers version is picked up from a clean import state
import os, gc, torch, time, json, numpy as np
from pathlib import Path

SMASH_SCRIPT = '''import sys, time, gc, json, numpy as np
from pathlib import Path
import torch

sys.path.insert(0, 'src')
from sdxl_opt.utils import seed_everything, reset_peak_memory, gpu_peak_memory_gb
from sdxl_opt.evaluate import compute_clip_scores

from diffusers import StableDiffusionXLPipeline
from pruna import smash, SmashConfig

print("Loading base SDXL for smash...")
base_pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, variant="fp16",
).to("cuda")

smash_config = SmashConfig()
smash_config["quantizer"] = "hqq_diffusers"
smash_config["cacher"]    = "deepcache"

t0 = time.perf_counter()
smashed_pipe = smash(model=base_pipe, smash_config=smash_config)
print(f"smash() done in {time.perf_counter()-t0:.1f}s")

prompts = [
    "a photo of an astronaut riding a horse on mars, cinematic lighting",
    "a beautiful sunset over a calm ocean with sailboats, oil painting",
    "a corgi wearing a top hat and monocle, portrait photography",
    "a futuristic cityscape at night with neon lights, cyberpunk",
]

# Warmup
gen = seed_everything(0)
_ = smashed_pipe("warmup", num_inference_steps=50, guidance_scale=7.5,
                  height=512, width=512, generator=gen)
torch.cuda.synchronize()

latencies = []
imgs = []
for run in range(3):
    gen = seed_everything(42)
    reset_peak_memory()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    batch_imgs = [smashed_pipe(p, num_inference_steps=50, guidance_scale=7.5,
                               generator=seed_everything(42)).images[0]
                  for p in prompts]
    torch.cuda.synchronize()
    per_img = (time.perf_counter() - t0) / len(prompts)
    latencies.append(per_img)
    imgs = batch_imgs
    peak = gpu_peak_memory_gb()
    print(f"  Run {run+1}/3: {per_img:.2f}s/img | Peak VRAM: {peak:.2f} GB")

result = {
    "config": "pruna_smash",
    "label": "pruna(hqq+deepcache)",
    "steps": 50,
    "latency_mean_s": round(float(np.mean(latencies)), 3),
    "latency_std_s":  round(float(np.std(latencies)), 3),
    "peak_vram_gb":   round(gpu_peak_memory_gb(), 2),
}

# Save images
img_dir = Path("results/images/pruna_smash")
img_dir.mkdir(parents=True, exist_ok=True)
for i, img in enumerate(imgs):
    img.save(img_dir / f"sample_{i:02d}.png")

# Write result to a temp file so parent process can read it
with open("/tmp/pruna_smash_result.json", "w") as f:
    json.dump(result, f)

baseline_lat = 5.66
print(f"\n  → {result['latency_mean_s']}s ± {result['latency_std_s']}s | {result['peak_vram_gb']} GB")
print(f"  → Speedup vs baseline: {baseline_lat / result['latency_mean_s']:.2f}×")
print("DONE")
'''

if not already_done('pruna_smash'):
    # Write the script
    with open('/tmp/pruna_smash_runner.py', 'w') as f:
        f.write(SMASH_SCRIPT)

    print("=== Pruna smash() ===")
    print("Running in subprocess (isolated transformers env)...")
    print("Expected time: ~10 min on A100\n")

    proc = subprocess.run(
        [sys.executable, '/tmp/pruna_smash_runner.py'],
        capture_output=False,   # streams output live to notebook
        text=True,
        cwd='/content/sdxl-optimization'
    )

    if proc.returncode == 0:
        with open('/tmp/pruna_smash_result.json') as f:
            r = json.load(f)
        results.append(r)
        baseline_lat = next((x['latency_mean_s'] for x in results if x['config'] == 'baseline'), 5.66)
        print(f"\n✅ pruna_smash done: {r['latency_mean_s']}s | {r['peak_vram_gb']} GB | {baseline_lat/r['latency_mean_s']:.2f}× speedup")
        save_checkpoint(results, '8.4_pruna_smash')
    else:
        print(f"❌ Subprocess failed (return code {proc.returncode})")
        print("Tip: If you see transformers errors, run:\n  !pip install 'transformers>=4.40,<4.47' pruna --upgrade")
        print("Then restart runtime and re-run from Cell 8.0")
else:
    print('✓ pruna_smash already done')


In [ ]:
# ============================================================
# CELL 8.5: Full updated results table — all experiments
# Merges Section 2-7 results (from checkpoint) + new HQQ results
# ============================================================
from sdxl_opt.evaluate import compute_clip_scores
from PIL import Image

df = pd.DataFrame(results)

# Reload images from Drive for CLIP scoring where possible
if 'clip_score' not in df.columns:
    df['clip_score'] = float('nan')

missing_clip = df[df['clip_score'].isna()]['config'].tolist()
for cfg_name in missing_clip:
    img_dir = Path(f'results/images/{cfg_name}')
    if img_dir.exists():
        loaded = [Image.open(p) for p in sorted(img_dir.glob('*.png'))[:len(prompts)]]
        if loaded:
            mean, std, _ = compute_clip_scores(loaded, prompts[:len(loaded)])
            df.loc[df['config'] == cfg_name, 'clip_score'] = mean

# Speedup
baseline_lat = df.loc[df['config'] == 'baseline', 'latency_mean_s'].values
if len(baseline_lat) > 0:
    df['speedup'] = (baseline_lat[0] / df['latency_mean_s']).round(2)

# Display sorted
cols = ['config', 'latency_mean_s', 'latency_std_s', 'peak_vram_gb', 'speedup', 'clip_score']
display_cols = [c for c in cols if c in df.columns]
print('=== Full Results (all experiments) ===')
display(df[display_cols].sort_values('speedup', ascending=False))

df.to_csv('results/benchmark_results_full.csv', index=False)
print('\n💾 Saved full results to Google Drive')

In [ ]:
# ============================================================
# CELL 8.6: Updated Pareto plot including HQQ results
# ============================================================
from sdxl_opt.pareto import plot_pareto, plot_speedup_bar

if 'clip_score' in df.columns and df['clip_score'].notna().any():
    plot_pareto(df[df['clip_score'].notna()], 'results/pareto_frontier_full.png')
    display(IPImage('results/pareto_frontier_full.png', width=800))

plot_speedup_bar(df, output_path='results/speedup_bar_full.png')
display(IPImage('results/speedup_bar_full.png', width=700))

print('💾 Updated plots saved to Google Drive')
print('\n📋 Quantization comparison:')
q_df = df[df['config'].isin(['baseline', 'int8_unet', 'nf4_unet', 'hqq_4bit'])]
if len(q_df):
    display(q_df[display_cols])
    print('\n→ BitsAndBytes (int8/nf4) added overhead → slower')
    print('→ HQQ native INT4 kernels → faster + smaller')